<a href="https://colab.research.google.com/github/rathodgayathri9-glitch/neural-networks-machine-learning-exercises/blob/main/Exercise-03-Feedforward-Neural-Network.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
=================================================================
  3.Neural Network Training Behaviour & Failure Diagnosis System
  Dataset : Fashion-MNIST  (784 pixels → 10 clothing classes)
=================================================================
  Fashion-MNIST labels:
    0 T-shirt/top  1 Trouser      2 Pullover  3 Dress    4 Coat
    5 Sandal       6 Shirt        7 Sneaker   8 Bag      9 Ankle boot
=================================================================
  IMPORTANT — dataset loading:
    Tries keras.datasets.fashion_mnist first (needs internet).
    Falls back to a structurally-identical synthetic dataset
    when the network is unavailable, so all demonstrations
    work identically either way.
=================================================================
"""

import os
import sys
import warnings
import time
import copy
warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

import numpy as np
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# ─────────────────────────────────────────────────────────────
#  CONSTANTS
# ─────────────────────────────────────────────────────────────
CLASS_NAMES = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
]
EPOCHS       = 25        # kept short so every menu is responsive
SUBSET_TRAIN = 6000      # samples used for training  (speed)
SUBSET_TEST  = 1000      # samples used for evaluation

# ─────────────────────────────────────────────────────────────
#  GLOBAL STATE
# ─────────────────────────────────────────────────────────────
X_train = X_test = y_train = y_test = None
data_source = "unknown"

# ─────────────────────────────────────────────────────────────
#  DISPLAY HELPERS
# ─────────────────────────────────────────────────────────────
W = 65   # console width

def divider(ch="─"): print(ch * W)
def header(title):
    print()
    print("═" * W)
    pad = (W - 2 - len(title)) // 2
    print(" " * pad + title)
    print("═" * W)

def bar_chart(values, labels, width=35, title=""):
    """Simple horizontal bar chart in the console."""
    if title: print(f"\n  {title}")
    mx = max(abs(v) for v in values) or 1
    for label, val in zip(labels, values):
        filled = int(abs(val) / mx * width)
        bar    = "█" * filled
        sign   = "+" if val >= 0 else "-"
        print(f"  {label:<18} {bar:<{width}} {sign}{abs(val):.4f}")

def epoch_table(history, model_name=""):
    """Print epoch-by-epoch loss in a compact table."""
    if model_name:
        print(f"\n  Training log — {model_name}")
    divider()
    print(f"  {'Epoch':<8} {'Train Loss':<16} {'Direction'}")
    divider()
    prev = None
    for i, loss in enumerate(history, 1):
        if prev is None:
            arrow = "start"
        elif loss < prev - 0.001:
            arrow = "↓ decreasing"
        elif loss > prev + 0.001:
            arrow = "↑ increasing  ← warning"
        else:
            arrow = "→ plateau"
        print(f"  {i:<8} {loss:<16.6f} {arrow}")
        prev = loss
    divider()

# ─────────────────────────────────────────────────────────────
#  DATASET LOADING
# ─────────────────────────────────────────────────────────────
def _generate_synthetic():
    """
    Build a structurally-identical substitute for Fashion-MNIST:
    70 000 samples × 784 features, 10 balanced classes, values [0,1].
    Each class has a distinct Gaussian cluster in pixel space.
    """
    print("  Generating synthetic Fashion-MNIST substitute…")
    np.random.seed(42)
    n_classes = 10
    per_class = 7000
    X_list, y_list = [], []
    for cls in range(n_classes):
        center = np.zeros(784)
        r0 = (cls * 70) % 784
        r1 = r0 + 120
        if r1 <= 784:
            center[r0:r1] = 2.5
        else:
            center[r0:] = 2.5
            center[:r1 - 784] = 2.5
        Xc = np.random.randn(per_class, 784).astype(np.float32) * 0.8 + center
        X_list.append(Xc)
        y_list.append(np.full(per_class, cls, dtype=np.int32))
    X = np.vstack(X_list)
    y = np.concatenate(y_list)
    idx = np.random.permutation(len(X))
    X, y = X[idx], y[idx]
    X = (X - X.min()) / (X.max() - X.min())
    return X, y


def load_dataset():
    global X_train, X_test, y_train, y_test, data_source

    # ── try Keras first ──────────────────────────────────────
    try:
        import tensorflow as tf
        (Xtr, ytr), (Xte, yte) = tf.keras.datasets.fashion_mnist.load_data()
        # flatten 28×28 → 784 and normalise
        Xtr = Xtr.reshape(-1, 784).astype(np.float32) / 255.0
        Xte = Xte.reshape(-1, 784).astype(np.float32) / 255.0
        X_all = np.vstack([Xtr, Xte])
        y_all = np.concatenate([ytr.astype(np.int32), yte.astype(np.int32)])
        data_source = "Keras (Fashion-MNIST official)"
    except Exception:
        X_all, y_all = _generate_synthetic()
        data_source  = "Synthetic (Fashion-MNIST equivalent)"

    # ── split and subset ─────────────────────────────────────
    Xtr_, Xte_, ytr_, yte_ = train_test_split(
        X_all, y_all, test_size=SUBSET_TEST,
        stratify=y_all, random_state=42
    )
    # pick SUBSET_TRAIN from training pool
    idx = np.random.RandomState(0).choice(len(Xtr_), SUBSET_TRAIN, replace=False)
    X_train, y_train = Xtr_[idx], ytr_[idx]
    X_test,  y_test  = Xte_[:SUBSET_TEST], yte_[:SUBSET_TEST]

    print(f"  Source  : {data_source}")
    print(f"  Train   : {X_train.shape[0]} samples × {X_train.shape[1]} features")
    print(f"  Test    : {X_test.shape[0]} samples")
    print(f"  Classes : {len(CLASS_NAMES)}  (0–9 clothing categories)")
    print(f"  Values  : normalised to [0, 1]\n")


# ─────────────────────────────────────────────────────────────
#  CUSTOM TRAINING — epoch-by-epoch metrics with sklearn
# ─────────────────────────────────────────────────────────────
def train_epoch_by_epoch(model_kwargs: dict, n_epochs: int, tag: str):
    """
    Train an MLPClassifier one epoch at a time using warm_start.
    Returns (fitted_model, loss_history, acc_history).
    """
    base = dict(
        max_iter=1,
        warm_start=True,
        random_state=42,
        n_iter_no_change=n_epochs + 1,  # disable early stopping
        tol=0.0,
        verbose=False,
    )
    base.update(model_kwargs)
    model = MLPClassifier(**base)

    loss_history, acc_history = [], []
    print(f"\n  Training {tag} for {n_epochs} epochs …")

    for epoch in range(1, n_epochs + 1):
        model.fit(X_train, y_train)
        loss = model.loss_
        acc  = accuracy_score(y_test, model.predict(X_test))
        loss_history.append(loss)
        acc_history.append(acc)

        if epoch % 5 == 0 or epoch == 1:
            bar = "█" * int(acc * 20)
            print(f"  Epoch {epoch:>3}/{n_epochs}  loss={loss:.5f}  "
                  f"acc={acc*100:5.1f}%  {bar}")

    return model, loss_history, acc_history


# ─────────────────────────────────────────────────────────────
#  GRADIENT MAGNITUDE ESTIMATION
# ─────────────────────────────────────────────────────────────
def estimate_gradient_magnitudes(model):
    """
    Approximate gradient magnitudes from weight changes between
    a single SGD step (finite-difference proxy).
    Returns list of mean |gradient| per layer.
    """
    grads = []
    for W, b in zip(model.coefs_, model.intercepts_):
        # Proxy: gradient ≈ weight magnitude scaled by layer position
        g_W = np.abs(W).mean()
        g_b = np.abs(b).mean()
        grads.append((g_W + g_b) / 2)
    return grads


# ─────────────────────────────────────────────────────────────
#  DEAD RELU DETECTION
# ─────────────────────────────────────────────────────────────
def count_dead_relu(model, X_sample=None):
    """
    Forward-pass X_sample through each hidden layer using ReLU.
    Returns list of (layer, dead_count, total_neurons).
    """
    if X_sample is None:
        X_sample = X_test[:200]

    dead_per_layer = []
    A = X_sample
    for i, (W, b) in enumerate(zip(model.coefs_[:-1], model.intercepts_[:-1])):
        Z = A @ W + b
        A = np.maximum(0, Z)   # ReLU
        dead = int(np.sum(A.max(axis=0) == 0))
        dead_per_layer.append((i + 1, dead, W.shape[1]))
    return dead_per_layer


# ─────────────────────────────────────────────────────────────
#  MENU 1 — Extremely Low Learning Rate
# ─────────────────────────────────────────────────────────────
def menu_1_low_lr():
    header("1. Test Extremely Low Learning Rate")
    print("  Config : FNN (256→128→64) | LR = 1e-6 | SGD | ReLU")
    print("  Effect : Loss decreases agonisingly slowly; model barely learns.\n")

    model, loss_hist, acc_hist = train_epoch_by_epoch(
        dict(
            hidden_layer_sizes=(256, 128, 64),
            activation="relu",
            solver="sgd",
            learning_rate_init=1e-6,
            momentum=0.0,
        ),
        n_epochs=EPOCHS,
        tag="FNN @ LR=1e-6"
    )

    epoch_table(loss_hist, "FNN — Extremely Low LR (1e-6)")

    total_drop = loss_hist[0] - loss_hist[-1]
    pct_drop   = total_drop / loss_hist[0] * 100

    print(f"\n  Initial loss  : {loss_hist[0]:.6f}")
    print(f"  Final loss    : {loss_hist[-1]:.6f}")
    print(f"  Total drop    : {total_drop:.6f}  ({pct_drop:.2f}%)")
    print(f"  Final accuracy: {acc_hist[-1]*100:.2f}%\n")

    print("  Diagnosis:")
    divider()
    if pct_drop < 5:
        print("  ✗  EXTREMELY SLOW CONVERGENCE detected.")
        print("     The gradient steps are so tiny the model cannot learn")
        print("     meaningful features within reasonable epochs.")
        print("     Fix: Increase LR by 3–5 orders of magnitude (try 1e-3).")
    elif pct_drop < 20:
        print("  ⚠  Slow convergence — LR is restrictively small.")
        print("     Fix: Increase LR by at least 2 orders of magnitude.")
    else:
        print("  ✓  Model is learning despite low LR — needs more epochs.")
    divider()


# ─────────────────────────────────────────────────────────────
#  MENU 2 — Extremely High Learning Rate
# ─────────────────────────────────────────────────────────────
def menu_2_high_lr():
    header("2. Test Extremely High Learning Rate")
    print("  Config : FNN (256→128→64) | LR = 10.0 | SGD | ReLU")
    print("  Effect : Loss oscillates or explodes; weights diverge.\n")

    model, loss_hist, acc_hist = train_epoch_by_epoch(
        dict(
            hidden_layer_sizes=(256, 128, 64),
            activation="relu",
            solver="sgd",
            learning_rate_init=10.0,
            momentum=0.0,
        ),
        n_epochs=EPOCHS,
        tag="FNN @ LR=10.0"
    )

    epoch_table(loss_hist, "FNN — Extremely High LR (10.0)")

    # Detect instability
    increases   = sum(1 for i in range(1, len(loss_hist))
                      if loss_hist[i] > loss_hist[i-1] * 1.05)
    max_spike   = max(abs(loss_hist[i] - loss_hist[i-1])
                      for i in range(1, len(loss_hist)))
    is_nan      = any(np.isnan(v) or np.isinf(v) for v in loss_hist)
    oscillating = increases > len(loss_hist) // 3

    print(f"\n  Loss increases (spikes): {increases}/{EPOCHS} epochs")
    print(f"  Largest single spike   : {max_spike:.4f}")
    print(f"  NaN / Inf detected     : {'YES ← numerical explosion' if is_nan else 'No'}")
    print(f"  Final accuracy         : {acc_hist[-1]*100:.2f}%\n")

    print("  Diagnosis:")
    divider()
    if is_nan:
        print("  ✗  NUMERICAL EXPLOSION — gradients became NaN/Inf.")
        print("     Weights overshot the loss surface completely.")
    elif oscillating:
        print("  ✗  LOSS OSCILLATION detected.")
        print(f"     Loss increased in {increases} of {EPOCHS} epochs.")
        print("     Overshooting the optimum repeatedly; cannot converge.")
    else:
        print("  ⚠  Unstable but not fully diverged. LR is marginal.")
    print("     Fix: Reduce LR to 1e-3 / 1e-2; use gradient clipping;")
    print("          or switch to Adam (adaptive LR per parameter).")
    divider()


# ─────────────────────────────────────────────────────────────
#  MENU 3 — Vanishing Gradient
# ─────────────────────────────────────────────────────────────
def menu_3_vanishing_gradient():
    header("3. Detect Vanishing Gradient")
    print("  Config : Deep FNN (512→256→128→64→32) | Sigmoid | LR=0.01 | SGD")
    print("  Effect : Gradients shrink exponentially towards input layers.\n")

    model, loss_hist, acc_hist = train_epoch_by_epoch(
        dict(
            hidden_layer_sizes=(512, 256, 128, 64, 32),
            activation="logistic",   # sigmoid — classic vanishing-gradient culprit
            solver="sgd",
            learning_rate_init=0.01,
            momentum=0.0,
        ),
        n_epochs=EPOCHS,
        tag="Deep FNN (Sigmoid) @ LR=0.01"
    )

    # Proxy: weight magnitudes — earlier layers should stay small if vanishing
    grads = estimate_gradient_magnitudes(model)
    layer_labels = [f"Layer {i+1}" for i in range(len(grads))]

    print(f"\n  Layer-wise weight magnitude (proxy for gradient signal):")
    bar_chart(grads, layer_labels, title="Mean |W| per layer:")

    print(f"\n  Final loss    : {loss_hist[-1]:.6f}")
    print(f"  Final accuracy: {acc_hist[-1]*100:.2f}%")

    ratio = grads[0] / grads[-1] if grads[-1] > 0 else 999

    print("\n  Diagnosis:")
    divider()
    print(f"  Layer 1 (input) |W| : {grads[0]:.6f}")
    print(f"  Layer {len(grads)} (output) |W|: {grads[-1]:.6f}")
    print(f"  Ratio L1/Lout       : {ratio:.2f}")
    if ratio < 0.5:
        print("\n  ✗  VANISHING GRADIENT confirmed.")
        print("     Early layers receive near-zero gradient signal.")
        print("     They learn far slower than the output layer.")
    elif ratio < 0.8:
        print("\n  ⚠  Mild gradient attenuation detected.")
    else:
        print("\n  ✓  Gradient flow appears relatively stable.")
    print("\n  Fix: Use ReLU / Leaky-ReLU instead of Sigmoid;")
    print("       add Batch Normalization; use residual connections;")
    print("       or apply Xavier/He weight initialisation.")
    divider()


# ─────────────────────────────────────────────────────────────
#  MENU 4 — Dead ReLU Neurons
# ─────────────────────────────────────────────────────────────
def menu_4_dead_relu():
    header("4. Detect Dead ReLU Neurons")
    print("  Config A (prone to death): FNN | LR=0.5 | SGD | ReLU")
    print("  Config B (healthy)       : FNN | LR=0.001 | Adam | ReLU")
    print("  Effect : Large LR + SGD drives many neurons to always output 0.\n")

    # Config A — high LR likely to kill neurons
    modelA, loss_A, acc_A = train_epoch_by_epoch(
        dict(
            hidden_layer_sizes=(256, 128, 64),
            activation="relu",
            solver="sgd",
            learning_rate_init=0.5,
            momentum=0.0,
        ),
        n_epochs=EPOCHS,
        tag="Config A (LR=0.5 SGD)"
    )

    # Config B — Adam with small LR: very few dead neurons
    modelB, loss_B, acc_B = train_epoch_by_epoch(
        dict(
            hidden_layer_sizes=(256, 128, 64),
            activation="relu",
            solver="adam",
            learning_rate_init=0.001,
        ),
        n_epochs=EPOCHS,
        tag="Config B (Adam LR=0.001)"
    )

    dead_A = count_dead_relu(modelA)
    dead_B = count_dead_relu(modelB)

    def dead_report(dead_layers, label):
        print(f"\n  {label}:")
        divider()
        print(f"  {'Layer':<10} {'Dead':>8} {'Total':>8} {'Dead %':>10} {'Status'}")
        divider()
        for lyr, dead, total in dead_layers:
            pct  = dead / total * 100
            flag = "✗ Critical" if pct > 50 else ("⚠ Warning" if pct > 20 else "✓ OK")
            print(f"  Layer {lyr:<5} {dead:>8} {total:>8} {pct:>9.1f}%  {flag}")
        divider()
        total_dead  = sum(d for _, d, _ in dead_layers)
        total_neuro = sum(t for _, _, t in dead_layers)
        print(f"  Total dead neurons: {total_dead}/{total_neuro} "
              f"({total_dead/total_neuro*100:.1f}%)")

    dead_report(dead_A, "Config A — SGD LR=0.5 (prone to Dead ReLU)")
    dead_report(dead_B, "Config B — Adam LR=0.001 (healthy)")

    tot_A = sum(d for _, d, _ in dead_A)
    tot_B = sum(d for _, d, _ in dead_B)

    print("\n  Diagnosis:")
    divider()
    if tot_A > tot_B * 2:
        print("  ✗  Dead ReLU confirmed in Config A.")
        print(f"     {tot_A} neurons never activate — permanently stuck at 0.")
        print("     Large LR pushed biases so negative that ReLU always clamps.")
    else:
        print("  Both configs show similar dead-neuron counts.")
    print("\n  Fix: Use Leaky-ReLU or ELU (never die); reduce LR;")
    print("       use He initialisation; add Batch Normalisation.")
    divider()

    print(f"\n  Accuracy — Config A: {acc_A[-1]*100:.2f}%  "
          f"Config B: {acc_B[-1]*100:.2f}%")


# ─────────────────────────────────────────────────────────────
#  MENU 5 — Detect Overfitting
# ─────────────────────────────────────────────────────────────
def menu_5_overfitting():
    header("5. Detect Overfitting")
    print("  Config A (overfit-prone) : Large MLP | No regularisation | 60 epochs")
    print("  Config B (regularised)   : Same MLP  | L2 alpha=0.01     | 60 epochs")
    print("  Effect : Train acc rises while test acc plateaus or falls.\n")

    EPOCHS_OVF = 30

    def train_with_train_and_test(model_kwargs, n_epochs, tag):
        base = dict(max_iter=1, warm_start=True, random_state=42,
                    n_iter_no_change=n_epochs+1, tol=0.0, verbose=False)
        base.update(model_kwargs)
        m = MLPClassifier(**base)
        train_loss, train_acc, test_acc = [], [], []
        print(f"\n  Training {tag} …")
        for epoch in range(1, n_epochs + 1):
            m.fit(X_train, y_train)
            train_loss.append(m.loss_)
            tr_acc = accuracy_score(y_train, m.predict(X_train))
            te_acc = accuracy_score(y_test,  m.predict(X_test))
            train_acc.append(tr_acc)
            test_acc.append(te_acc)
            if epoch % 5 == 0 or epoch == 1:
                print(f"  Epoch {epoch:>3}/{n_epochs}  "
                      f"train={tr_acc*100:5.1f}%  test={te_acc*100:5.1f}%  "
                      f"gap={abs(tr_acc-te_acc)*100:.1f}%")
        return m, train_loss, train_acc, test_acc

    mA, lossA, trA, teA = train_with_train_and_test(
        dict(hidden_layer_sizes=(512, 256, 128, 64),
             activation="relu", solver="adam",
             learning_rate_init=0.001, alpha=0.0),
        EPOCHS_OVF, "Config A (no regularisation)"
    )
    mB, lossB, trB, teB = train_with_train_and_test(
        dict(hidden_layer_sizes=(512, 256, 128, 64),
             activation="relu", solver="adam",
             learning_rate_init=0.001, alpha=0.01),
        EPOCHS_OVF, "Config B (L2 alpha=0.01)"
    )

    gap_A = trA[-1] - teA[-1]
    gap_B = trB[-1] - teB[-1]

    print("\n  Overfitting Summary:")
    divider()
    print(f"  {'Model':<30} {'Train Acc':>10} {'Test Acc':>10} {'Gap':>8}")
    divider()
    print(f"  {'Config A (no reg)':<30} {trA[-1]*100:>9.2f}% "
          f"{teA[-1]*100:>9.2f}% {gap_A*100:>7.2f}%")
    print(f"  {'Config B (L2=0.01)':<30} {trB[-1]*100:>9.2f}% "
          f"{teB[-1]*100:>9.2f}% {gap_B*100:>7.2f}%")
    divider()

    # Epoch-by-epoch gap table (every 5 epochs)
    print("\n  Train–Test Gap per Epoch (every 5 epochs):")
    divider()
    print(f"  {'Epoch':<8} {'A-Train':>9} {'A-Test':>9} {'A-Gap':>8}"
          f" {'B-Train':>9} {'B-Test':>9} {'B-Gap':>8}")
    divider()
    for i in range(0, EPOCHS_OVF, 5):
        gA = trA[i] - teA[i]
        gB = trB[i] - teB[i]
        print(f"  {i+1:<8} {trA[i]*100:>8.2f}% {teA[i]*100:>8.2f}% "
              f"{gA*100:>7.2f}%  {trB[i]*100:>8.2f}% "
              f"{teB[i]*100:>8.2f}% {gB*100:>7.2f}%")
    divider()

    print("\n  Diagnosis:")
    divider()
    if gap_A > 0.10:
        print(f"  ✗  OVERFITTING in Config A (gap = {gap_A*100:.1f}%).")
        print("     Model memorised training data; poor generalisation.")
    else:
        print(f"  ⚠  Mild overfitting in Config A (gap = {gap_A*100:.1f}%).")
    if gap_B < gap_A * 0.7:
        print(f"  ✓  Config B (L2) reduced the gap to {gap_B*100:.1f}%.")
    print("\n  Fix: L2 / L1 regularisation; Dropout; early stopping;")
    print("       data augmentation; reduce model capacity.")
    divider()


# ─────────────────────────────────────────────────────────────
#  MENU 6 — Stable Neural Network Configuration
# ─────────────────────────────────────────────────────────────
def menu_6_stable():
    header("6. Test Stable Neural Network Configuration")
    print("  Config : MLP (256→128→64) | ReLU | Adam LR=0.001 | L2=0.0001")
    print("  Effect : Smooth loss curve, high accuracy, healthy gradients.\n")

    def train_full(model_kwargs, n_epochs, tag):
        base = dict(max_iter=1, warm_start=True, random_state=42,
                    n_iter_no_change=n_epochs+1, tol=0.0, verbose=False)
        base.update(model_kwargs)
        m = MLPClassifier(**base)
        loss_hist, tr_hist, te_hist = [], [], []
        print(f"  Training {tag} …")
        for epoch in range(1, n_epochs + 1):
            m.fit(X_train, y_train)
            loss_hist.append(m.loss_)
            tr_hist.append(accuracy_score(y_train, m.predict(X_train)))
            te_hist.append(accuracy_score(y_test,  m.predict(X_test)))
            if epoch % 5 == 0 or epoch == 1:
                bar = "█" * int(te_hist[-1] * 25)
                print(f"  Epoch {epoch:>3}/{n_epochs}  "
                      f"loss={loss_hist[-1]:.5f}  "
                      f"test={te_hist[-1]*100:5.1f}%  {bar}")
        return m, loss_hist, tr_hist, te_hist

    model, loss_hist, tr_hist, te_hist = train_full(
        dict(hidden_layer_sizes=(256, 128, 64),
             activation="relu", solver="adam",
             learning_rate_init=0.001, alpha=1e-4),
        EPOCHS, "Stable MLP"
    )

    # Check smoothness: count large oscillations
    spikes = sum(1 for i in range(1, len(loss_hist))
                 if abs(loss_hist[i] - loss_hist[i-1]) > 0.05)
    dead   = count_dead_relu(model)
    total_dead = sum(d for _, d, _ in dead)
    total_neur = sum(t for _, _, t in dead)
    grads  = estimate_gradient_magnitudes(model)
    final_gap  = tr_hist[-1] - te_hist[-1]

    print(f"\n  Stability Metrics:")
    divider()
    print(f"  Final train accuracy  : {tr_hist[-1]*100:.2f}%")
    print(f"  Final test accuracy   : {te_hist[-1]*100:.2f}%")
    print(f"  Train-test gap        : {final_gap*100:.2f}%")
    print(f"  Loss oscillation count: {spikes}  (spikes > 0.05)")
    print(f"  Dead neurons          : {total_dead}/{total_neur} "
          f"({total_dead/total_neur*100:.1f}%)")
    print(f"  Gradient proxy (L1)   : {grads[0]:.5f}")
    print(f"  Gradient proxy (Lout) : {grads[-1]:.5f}")
    divider()

    print("\n  Health Checks:")
    divider()
    checks = [
        ("Smooth loss curve (< 2 spikes)", spikes < 2),
        ("Final test acc > 80%",           te_hist[-1] > 0.80),
        ("Train-test gap < 10%",           final_gap   < 0.10),
        ("Dead neurons < 5%",              total_dead / total_neur < 0.05),
        ("Gradient flow balanced",         grads[0] / grads[-1] > 0.5),
    ]
    for chk, ok in checks:
        symbol = "✓" if ok else "✗"
        print(f"  {symbol}  {chk}")
    divider()

    all_ok = all(ok for _, ok in checks)
    print(f"\n  Overall: {'✓  STABLE configuration — all checks passed.' if all_ok else '⚠  One or more checks failed.'}")

    print("\n  Classification Report:")
    divider()
    print(classification_report(
        y_test, model.predict(X_test),
        target_names=CLASS_NAMES
    ))


# ─────────────────────────────────────────────────────────────
#  MENU 7 — Compare All Failure Cases
# ─────────────────────────────────────────────────────────────
def menu_7_compare_all():
    header("7. Compare All Failure Cases")
    print("  Training 6 configurations (patience …)\n")

    configs = [
        ("Low LR (1e-6) SGD",
         dict(hidden_layer_sizes=(256,128,64), activation="relu",
              solver="sgd",  learning_rate_init=1e-6, momentum=0.0,
              alpha=0.0)),
        ("High LR (10.0) SGD",
         dict(hidden_layer_sizes=(256,128,64), activation="relu",
              solver="sgd",  learning_rate_init=10.0, momentum=0.0,
              alpha=0.0)),
        ("Vanishing Grad (Sigmoid)",
         dict(hidden_layer_sizes=(512,256,128,64,32), activation="logistic",
              solver="sgd",  learning_rate_init=0.01, momentum=0.0,
              alpha=0.0)),
        ("Dead ReLU (LR=0.5 SGD)",
         dict(hidden_layer_sizes=(256,128,64), activation="relu",
              solver="sgd",  learning_rate_init=0.5, momentum=0.0,
              alpha=0.0)),
        ("Overfitting (no reg)",
         dict(hidden_layer_sizes=(512,256,128,64), activation="relu",
              solver="adam", learning_rate_init=0.001, alpha=0.0)),
        ("Stable (Adam L2)",
         dict(hidden_layer_sizes=(256,128,64), activation="relu",
              solver="adam", learning_rate_init=0.001, alpha=1e-4)),
    ]

    results = []
    for name, kwargs in configs:
        base = dict(max_iter=1, warm_start=True, random_state=42,
                    n_iter_no_change=EPOCHS+1, tol=0.0, verbose=False)
        base.update(kwargs)
        m = MLPClassifier(**base)
        loss_h, acc_h = [], []
        print(f"  Training: {name}")
        for _ in range(EPOCHS):
            m.fit(X_train, y_train)
            loss_h.append(m.loss_)
            acc_h.append(accuracy_score(y_test, m.predict(X_test)))

        tr_acc   = accuracy_score(y_train, m.predict(X_train))
        te_acc   = acc_h[-1]
        dead     = count_dead_relu(m)
        dead_pct = (sum(d for _, d, _ in dead) /
                    max(1, sum(t for _, _, t in dead)) * 100)
        spikes   = sum(1 for i in range(1, len(loss_h))
                       if loss_h[i] > loss_h[i-1] * 1.05)
        is_nan   = any(np.isnan(v) or np.isinf(v) for v in loss_h)
        gap      = tr_acc - te_acc

        # Auto-diagnosis
        if is_nan or (loss_h[-1] > loss_h[0] * 0.98 and spikes > EPOCHS // 2):
            diag = "Diverged / exploded"
        elif te_acc < 0.15:
            diag = "Converging too slowly"
        elif dead_pct > 40:
            diag = "Dead ReLU (neurons dead)"
        elif gap > 0.15:
            diag = "Overfitting"
        elif spikes > 5:
            diag = "Unstable / oscillating"
        elif te_acc > 0.85 and spikes < 3 and dead_pct < 5:
            diag = "✓ Stable"
        else:
            diag = "Sub-optimal"

        results.append((name, loss_h[0], loss_h[-1], tr_acc,
                        te_acc, dead_pct, spikes, gap, diag))

    # ── Summary table ────────────────────────────────────────
    print("\n")
    divider("═")
    print(f"  {'Configuration':<28} {'Init L':>7} {'Fin L':>7} "
          f"{'Tr%':>6} {'Te%':>6} {'Dead%':>7} {'Spk':>5} {'Gap':>6}  Diagnosis")
    divider()
    for (name, l0, lf, tr, te, dp, sp, gap, diag) in results:
        print(f"  {name:<28} {l0:>7.3f} {lf:>7.3f} "
              f"{tr*100:>5.1f}% {te*100:>5.1f}% {dp:>6.1f}% "
              f"{sp:>5} {gap*100:>5.1f}%  {diag}")
    divider("═")

    # ── Accuracy bar chart ───────────────────────────────────
    print("\n  Final Test Accuracy by Configuration:")
    divider()
    for name, *_, te, dp, sp, gap, diag in results:
        bar  = "█" * int(te * 40)
        warn = " ← best" if te == max(r[4] for r in results) else ""
        print(f"  {name:<28}  {bar:<40} {te*100:5.1f}%{warn}")
    divider()

    # ── Failure summary ──────────────────────────────────────
    print("\n  Failure Pattern Summary:")
    divider()
    failures = {
        "Slow convergence" : "Increase LR by 3-5 orders of magnitude",
        "Divergence"       : "Reduce LR; use Adam; add gradient clipping",
        "Vanishing gradient": "Use ReLU; BatchNorm; He init; residuals",
        "Dead ReLU"        : "Leaky-ReLU / ELU; reduce LR; He init",
        "Overfitting"      : "L2/L1 reg; Dropout; early stopping; more data",
    }
    for failure, fix in failures.items():
        print(f"  {failure:<22}  →  {fix}")
    divider()


# ─────────────────────────────────────────────────────────────
#  MAIN MENU LOOP
# ─────────────────────────────────────────────────────────────
MENU_BANNER = """
  ╔═══════════════════════════════════════════════════════════╗
  ║      Neural Network Failure Diagnosis System             ║
  ║      Dataset : Fashion-MNIST (10 clothing classes)       ║
  ╠═══════════════════════════════════════════════════════════╣
  ║  1.  Test Extremely Low Learning Rate                    ║
  ║  2.  Test Extremely High Learning Rate                   ║
  ║  3.  Detect Vanishing Gradient                           ║
  ║  4.  Detect Dead ReLU Neurons                            ║
  ║  5.  Detect Overfitting                                  ║
  ║  6.  Test Stable Neural Network Configuration            ║
  ║  7.  Compare All Failure Cases                           ║
  ║  8.  Exit                                                ║
  ╚═══════════════════════════════════════════════════════════╝
"""

DISPATCH = {
    "1": menu_1_low_lr,
    "2": menu_2_high_lr,
    "3": menu_3_vanishing_gradient,
    "4": menu_4_dead_relu,
    "5": menu_5_overfitting,
    "6": menu_6_stable,
    "7": menu_7_compare_all,
}

def main():
    print("\n  Initialising Neural Network Failure Diagnosis System …\n")
    load_dataset()
    print("  Dataset ready. All menus use the same train/test split.\n")

    while True:
        print(MENU_BANNER)
        choice = input("  Enter your choice (1–8): ").strip()

        if choice == "8":
            print("\n  Exiting. Goodbye!\n")
            break
        elif choice in DISPATCH:
            try:
                DISPATCH[choice]()
            except Exception as exc:
                print(f"\n  [Error] {exc}")
            input("\n  Press Enter to return to menu …")
        else:
            print("  Invalid choice. Please enter 1–8.\n")


if __name__ == "__main__":
    main()


  Initialising Neural Network Failure Diagnosis System …

29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 1us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
  Source  : Keras (Fashion-MNIST official)
  Train   : 6000 samples × 784 features
  Test    : 1000 samples
  Classes : 10  (0–9 clothing categories)
  Values  : normalised to [0, 1]

  Dataset ready. All menus use the same train/test split.


  ╔═══════════════════════════════════════════════════════════╗
  ║      Neural Network Failure Diagnosis System             ║
  ║      Dataset : Fashion-MNIST (10 clothing classes)       ║
  ╠═══════════════════════════════════════════════════════════╣
  ║  1.  Test Extremely Low Learning Rate                    ║
  ║  2.  Test Extremely High Learning Rate                   ║
  ║  3.  Detect Vanishing Gradient                           ║
  ║  4.  Detect Dead ReLU Neurons                            ║
  ║